
# COMM117 Workshop 10 — Advanced RAG, Chaining, Tool Use, and Routing

This workshop covers the last two lectures:
- RAG evaluation and diagnostics
- Advanced / modular RAG, variants, chaining, tool use, routing, and corrective verification

This **exercise notebook includes the toy documents and structured data inside the notebook**, so students do not need any extra files.



## Learning goals

By the end of this workshop, you should be able to:
1. Retrieve relevant chunks from a small corpus
2. Evaluate retrieval with hit rate, precision\@k, recall\@k, and reciprocal rank
3. Improve baseline RAG with query rewriting and metadata filtering
4. Implement simple multi-hop retrieval
5. Distinguish RAG from tool use
6. Route a query to the correct subsystem
7. Add a corrective verification step


In [ ]:

# Basic imports
import math
import re
from collections import Counter

import pandas as pd


## 0. Toy knowledge base and structured data

In [ ]:

# Unstructured documents used throughout the workshop
DOCUMENTS = [
    {
        "doc_id": "policy_1",
        "title": "Refund Policy",
        "department": "support",
        "region": "global",
        "text": (
            "Refunds are available within 14 days of purchase for physical products in original condition. "
            "Digital products are non-refundable once downloaded. "
            "If a customer was charged twice, the duplicate charge can be refunded after verification. "
            "Subscription renewals are refundable only within 48 hours of renewal."
        ),
    },
    {
        "doc_id": "faq_1",
        "title": "Account FAQ",
        "department": "support",
        "region": "global",
        "text": (
            "Customers can reset their password using the account settings page. "
            "Two-factor authentication can be enabled in security settings. "
            "If login fails with error code 0xA93, the issue is usually related to authentication failure."
        ),
    },
    {
        "doc_id": "shipping_1",
        "title": "Shipping Policy",
        "department": "operations",
        "region": "global",
        "text": (
            "Standard shipping takes 3 to 5 business days. "
            "Express shipping takes 1 to 2 business days. "
            "Shipping fees are non-refundable after the order has been dispatched."
        ),
    },
    {
        "doc_id": "billing_1",
        "title": "Billing and Subscription",
        "department": "finance",
        "region": "global",
        "text": (
            "Monthly subscriptions renew automatically unless cancelled before the renewal date. "
            "Users can cancel from the billing settings page. "
            "Refund requests for subscription renewals must be submitted within 48 hours."
        ),
    },
    {
        "doc_id": "hr_uk_1",
        "title": "UK Leave Policy",
        "department": "hr",
        "region": "uk",
        "text": (
            "Full-time staff in the UK receive 25 days of annual leave per year, excluding public holidays. "
            "Unused leave may be carried over only with manager approval."
        ),
    },
    {
        "doc_id": "hr_us_1",
        "title": "US Leave Policy",
        "department": "hr",
        "region": "us",
        "text": (
            "Full-time staff in the US receive 15 days of paid time off per year. "
            "Public holidays are managed separately from PTO."
        ),
    },
    {
        "doc_id": "movie_1",
        "title": "The Dalton Girls",
        "department": "wiki",
        "region": "global",
        "text": "The Dalton Girls is a film directed by Reginald Le Borg.",
    },
    {
        "doc_id": "person_1",
        "title": "Reginald Le Borg",
        "department": "wiki",
        "region": "global",
        "text": "Reginald Le Borg died from a heart attack.",
    },
]

# Structured data used for tool-use questions
SALES_TABLE = {
    "Q1": {"actual": 100, "forecast": 95},
    "Q2": {"actual": 120, "forecast": 110},
    "Q3": {"actual": 105, "forecast": 108},
    "Q4": {"actual": 130, "forecast": 125},
}

pd.DataFrame(DOCUMENTS)[["doc_id", "title", "department", "region"]]


## 1. Chunk the documents

In [ ]:

def sentence_chunk(text, max_words=40):
    """Simple sentence-based chunking for the toy dataset."""
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    chunks = []
    current = []

    for sent in sentences:
        tentative = " ".join(current + [sent])
        if len(tentative.split()) <= max_words:
            current.append(sent)
        else:
            if current:
                chunks.append(" ".join(current))
            current = [sent]

    if current:
        chunks.append(" ".join(current))
    return chunks


In [ ]:

# TODO: build a chunk table called chunks_df.
# Each row should contain: chunk_id, doc_id, title, department, region, text
# Use sentence_chunk(doc["text"], max_words=40)

chunk_rows = []

# YOUR CODE HERE
for doc in DOCUMENTS:
    chunks = sentence_chunk(doc["text"], max_words=40)
    for i, chunk in enumerate(chunks):
        chunk_rows.append({
            "chunk_id": f'{doc["doc_id"]}_chunk_{i}',
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "department": doc["department"],
            "region": doc["region"],
            "text": chunk,
        })

chunks_df = pd.DataFrame(chunk_rows)
chunks_df


## 2. Baseline lexical retrieval

In [ ]:

def tokenize(text):
    return re.findall(r"[a-zA-Z0-9_]+", text.lower())


def simple_retrieve(query, chunks_df, top_k=3):
    """A tiny lexical retriever using token overlap."""
    q_tokens = set(tokenize(query))
    scored = []

    for _, row in chunks_df.iterrows():
        c_tokens = set(tokenize(row["text"]))
        score = len(q_tokens & c_tokens)
        item = row.to_dict()
        item["score"] = score
        scored.append(item)

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


In [ ]:

# TODO: test the baseline retriever on a refund question.
results = simple_retrieve("Can I get a refund for a downloaded digital product?", chunks_df, top_k=3)

for r in results:
    print(f"[{r['score']}] {r['chunk_id']} | {r['title']}")
    print(r["text"])
    print()


## 3. Retrieval evaluation

In [ ]:

EVAL_SET = [
    {
        "question": "Can I get a refund for a downloaded digital product?",
        "gold_chunk_ids": ["policy_1_chunk_0"],
    },
    {
        "question": "What does error code 0xA93 usually mean?",
        "gold_chunk_ids": ["faq_1_chunk_0"],
    },
    {
        "question": "How long does express shipping take?",
        "gold_chunk_ids": ["shipping_1_chunk_0"],
    },
    {
        "question": "How many days of annual leave do UK full-time staff receive?",
        "gold_chunk_ids": ["hr_uk_1_chunk_0"],
    },
]


In [ ]:

# TODO: implement retrieval metrics.
def hit_rate_at_k(results, gold_ids):
    retrieved_ids = [r["chunk_id"] for r in results]
    return int(any(g in retrieved_ids for g in gold_ids))


def precision_at_k(results, gold_ids, k):
    retrieved_ids = [r["chunk_id"] for r in results[:k]]
    relevant = sum(1 for rid in retrieved_ids if rid in gold_ids)
    return relevant / k


def recall_at_k(results, gold_ids, k):
    retrieved_ids = [r["chunk_id"] for r in results[:k]]
    relevant = sum(1 for g in gold_ids if g in retrieved_ids)
    return relevant / len(gold_ids)


def reciprocal_rank(results, gold_ids):
    retrieved_ids = [r["chunk_id"] for r in results]
    for rank, rid in enumerate(retrieved_ids, start=1):
        if rid in gold_ids:
            return 1 / rank
    return 0.0


In [ ]:

rows = []
for ex in EVAL_SET:
    results = simple_retrieve(ex["question"], chunks_df, top_k=3)
    rows.append({
        "question": ex["question"],
        "hit@3": hit_rate_at_k(results, ex["gold_chunk_ids"]),
        "precision@3": precision_at_k(results, ex["gold_chunk_ids"], 3),
        "recall@3": recall_at_k(results, ex["gold_chunk_ids"], 3),
        "MRR": reciprocal_rank(results, ex["gold_chunk_ids"]),
    })

pd.DataFrame(rows)


## 4. Query-transformed RAG

In [ ]:

def rewrite_query_simple(query):
    q = query.lower()
    q = q.replace("money back", "refund")
    q = q.replace("downloaded file", "digital product")
    q = q.replace("leave policy", "annual leave policy")
    return q


In [ ]:

# TODO: compare original and rewritten retrieval.
query = "Can I get my money back for a downloaded file?"
rewritten = rewrite_query_simple(query)

print("ORIGINAL:", query)
print("REWRITTEN:", rewritten)
print()

print("TOP RESULTS AFTER REWRITE")
for r in simple_retrieve(rewritten, chunks_df, top_k=3):
    print(f"[{r['score']}] {r['chunk_id']} | {r['title']}")


## 5. Filtered RAG

In [ ]:

def filtered_retrieve(query, chunks_df, top_k=3, region=None, department=None):
    subset = chunks_df.copy()
    if region is not None:
        subset = subset[subset["region"] == region]
    if department is not None:
        subset = subset[subset["department"] == department]
    return simple_retrieve(query, subset, top_k=top_k)


In [ ]:

# TODO: retrieve the UK leave policy using a region filter.
results = filtered_retrieve(
    "What is the annual leave policy?",
    chunks_df,
    top_k=3,
    region="uk",
    department="hr",
)

for r in results:
    print(f"[{r['score']}] {r['chunk_id']} | {r['title']} | {r['region']}")
    print(r["text"])
    print()


## 6. Parent-child / hierarchical retrieval

In [ ]:

# In this toy notebook, the child chunk is the retrieved chunk.
# The parent document is the full source document.

def parent_document_lookup(doc_id):
    for doc in DOCUMENTS:
        if doc["doc_id"] == doc_id:
            return doc
    return None


In [ ]:

# TODO: retrieve a small chunk, then map it back to the full parent document.
child_results = simple_retrieve("digital products refund", chunks_df, top_k=1)
child = child_results[0]
parent_doc = parent_document_lookup(child["doc_id"])

print("CHILD CHUNK")
print(child["chunk_id"], "|", child["text"])
print()
print("PARENT DOCUMENT")
print(parent_doc["title"])
print(parent_doc["text"])


## 7. Multi-hop RAG

In [ ]:

def multi_hop_rag(question):
    # Hop 1: find the director of The Dalton Girls
    hop1 = simple_retrieve("The Dalton Girls director", chunks_df, top_k=1)[0]

    # Extract the intermediate entity manually for this teaching example
    director = "Reginald Le Borg"

    # Hop 2: search for how that person died
    hop2 = simple_retrieve(f"{director} died", chunks_df, top_k=1)[0]

    answer = "Heart attack."
    return {
        "question": question,
        "hop1": hop1,
        "intermediate_entity": director,
        "hop2": hop2,
        "answer": answer,
    }


In [ ]:

# TODO: run the multi-hop workflow.
out = multi_hop_rag("Why did the director of The Dalton Girls die?")
print("QUESTION:", out["question"])
print("HOP 1:", out["hop1"]["text"])
print("INTERMEDIATE ENTITY:", out["intermediate_entity"])
print("HOP 2:", out["hop2"]["text"])
print("FINAL ANSWER:", out["answer"])


## 8. Tool use

In [ ]:

def calculator_tool(expression):
    return eval(expression, {"__builtins__": {}}, {})


def sales_delta_tool(quarter):
    row = SALES_TABLE[quarter]
    delta = row["actual"] - row["forecast"]
    return {
        "quarter": quarter,
        "actual": row["actual"],
        "forecast": row["forecast"],
        "delta": delta,
    }


In [ ]:

# TODO: test the tools.
print("Calculator tool:", calculator_tool("90 - 20 - 35"))
print("Sales tool:", sales_delta_tool("Q2"))


## 9. Router

In [ ]:

def route_query(query):
    q = query.lower()

    if any(x in q for x in ["sales", "forecast", "revenue", "q1", "q2", "q3", "q4"]):
        return "sales_tool"
    if any(ch.isdigit() for ch in q) and any(sym in q for sym in ["+", "-", "*", "/"]):
        return "calculator"
    return "rag"


In [ ]:

# TODO: inspect the router decisions.
queries = [
    "Can I get a refund for a downloaded digital product?",
    "Q2 actual versus forecast",
    "90 - 20 - 35",
]

for q in queries:
    print(q, "->", route_query(q))


## 10. Corrective / verification step

In [ ]:

def draft_answer_from_context(query, retrieved):
    context = " ".join(r["text"] for r in retrieved)
    if "digital products are non-refundable once downloaded" in context.lower():
        return "No. Digital products are non-refundable once downloaded."
    if "error code 0xa93" in context.lower():
        return "Error code 0xA93 usually indicates an authentication failure."
    return "Not enough information."


def verify_answer(answer, retrieved):
    context = " ".join(r["text"] for r in retrieved).lower()
    if answer.lower() == "not enough information.":
        return True
    important_words = [w for w in tokenize(answer) if len(w) > 3]
    overlap = sum(1 for w in important_words if w in context)
    return overlap >= max(1, len(important_words) // 3)


In [ ]:

# TODO: retrieve, draft, verify.
retrieved = simple_retrieve("Can I refund a downloaded digital product?", chunks_df, top_k=2)
draft = draft_answer_from_context("Can I refund a downloaded digital product?", retrieved)
supported = verify_answer(draft, retrieved)

print("DRAFT:", draft)
print("SUPPORTED:", supported)


## 11. End-to-end chained workflow

In [ ]:

def advanced_rag_workflow(query):
    route = route_query(query)

    if route == "calculator":
        result = calculator_tool(query)
        return {
            "route": route,
            "answer": result,
        }

    if route == "sales_tool":
        quarter = next((q for q in SALES_TABLE.keys() if q.lower() in query.lower()), None)
        result = sales_delta_tool(quarter)
        return {
            "route": route,
            "answer": f"{quarter}: actual={result['actual']}, forecast={result['forecast']}, delta={result['delta']}",
        }

    rewritten = rewrite_query_simple(query)
    retrieved = simple_retrieve(rewritten, chunks_df, top_k=2)
    draft = draft_answer_from_context(query, retrieved)
    supported = verify_answer(draft, retrieved)
    final_answer = draft if supported else "Not enough information."

    return {
        "route": route,
        "rewritten_query": rewritten,
        "retrieved": retrieved,
        "draft": draft,
        "supported": supported,
        "answer": final_answer,
    }


In [ ]:

# TODO: run the full workflow on three different query types.
for q in [
    "Can I get my money back for a downloaded file?",
    "Q2 actual versus forecast",
    "90 - 20 - 35",
]:
    print("=" * 80)
    print("QUERY:", q)
    out = advanced_rag_workflow(q)
    for k, v in out.items():
        print(k, ":", v)



## 12. Reflection questions

1. Which questions are best handled by basic retrieval?
2. Which questions benefit from query transformation?
3. Why does filtered RAG help with the leave-policy example?
4. Why is the Dalton Girls question multi-hop?
5. Which questions require tool use rather than document retrieval?
6. If a final answer is wrong, which module failed: routing, retrieval, tool use, or verification?
